In [17]:
import re, unicodedata, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, BertTokenizerFast

# Text Encoder：把一篇貼文變成一串數字

電腦看不懂文字，只看得懂數字。**Text Encoder 的工作，就是把每篇貼文的文字轉成一串固定長度的數字（向量）。** 在這份程式裡，每篇貼文都會變成 **768 個數字**。

你可以把這 768 個數字想成貼文在「語意地圖」上的座標：
- 內容相似的貼文（例如兩篇都在推保健食品），座標會靠得比較近
- 內容差很多的貼文（推保健食品 vs. 分享旅遊心得），座標會離得比較遠

之後的分類器（判斷是不是業配）就是看著這些座標做判斷。

**整體流程**

```
貼文文字 ──► 清理 + 刪業配詞 ──► 切成 token ──► BERT（12 層）──► 取 [CLS] 位置的向量 ──► X^T（768 維）
```

| 套件 | 白話解釋 | 在這份程式的用途 |
|---|---|---|
| `unicodedata` | 處理 Unicode 字元 | 把全形字轉成半形（`＃`→`#`） |
| `pathlib.Path` | 用物件的方式處理檔案路徑 | 組出 csv 路徑 |
| `torch`（PyTorch） | 深度學習框架，負責張量（tensor）運算和 GPU 加速 | 跑 BERT 模型 |
| `transformers` | Hugging Face 出的套件，可以一行下載別人訓練好的語言模型 | 載入 BERT 和 tokenizer |

> **小字典｜tensor（張量）**：多維度的數字陣列。1 維是一串數字，2 維是表格，3 維是一疊表格。PyTorch 裡所有資料都是 tensor。

## 參數設定

| 參數 | 意思 |
|---|---|
| `MODEL_NAME` | 要用哪個預訓練模型。`ckiplab/bert-base-chinese` 是中研院 CKIP 實驗室用**繁體中文**語料訓練的 BERT |
| `MAX_LEN = 512` | 一篇貼文最多吃 512 個 token，超過的部分直接切掉。這是 BERT 的硬性上限 |
| `BATCH_SIZE = 32` | 一次餵 32 篇貼文給模型。一次餵一篇太慢，一次餵全部記憶體會爆，所以分批 |
| `APPLY_MASK` | 要不要刪掉業配詞。設成 `False` 就是對照組（ablation） |
| `device` | 有 GPU（`cuda`）就用 GPU，沒有就用 CPU。GPU 很擅長同時做大量乘法，跑模型通常快 10 倍以上 |

> **小字典｜預訓練（pre-trained）**：別人已經用大量文字把模型訓練好了，我們直接拿來用，不用從零開始教它認識中文。就像請一個已經讀完很多書的人來幫忙讀貼文。
>
> **小字典｜ablation（消融實驗）**：故意拿掉某個步驟，看結果差多少，藉此證明這個步驟有沒有用。

In [18]:
PILOT_CSV = Path("/home/yingtzudai/Multimodal_SPoD_Video/data/split/pilot_posts.csv")
META_DIR  = Path("/home/ftp_246/data_5/多模態2026_ig_top200/Metadata_by_Influencer")
OUT_DIR   = Path("/home/yingtzudai/Multimodal_SPoD_Video/features/text")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "ckiplab/bert-base-chinese"   # 對照組可改 "xlm-roberta-base"
MAX_LEN    = 512
BATCH_SIZE = 32
APPLY_MASK = True                            # 跑 ablation 時改 False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


## 讀取資料：用 media_id 找回貼文文字

Pilot 的 csv 裡只有貼文編號和標籤，沒有文字。文字存在每位網紅各自的 metadata 檔裡，所以要做一次「查表」：

1. 對每位網紅，打開 `網紅帳號-FullVideoInfo.csv`
2. 建一本字典 `lookup`：`media_id → 貼文文字`
3. 用 pilot 裡的 `media_id` 去字典查文字（`.map(lookup)`）

**為什麼 media_id 要讀成字串（`dtype=str`）？**
IG 的 id 是很長的數字，例如 `3512345678901234567`。pandas 如果把它當成數字讀，會存成浮點數，尾巴幾位會失真，變成 `3512345678901234432` 之類的，結果就對不上了。當成文字讀就不會出這種事。

`.fillna("")`：如果有貼文找不到文字（NaN，空值），就用空字串代替，避免後面的程式出錯。

In [19]:
# media_id 一律讀成字串：IG 的 id 很長，讀成數字會因浮點精度失真而對不上
pilot = pd.read_csv(PILOT_CSV, dtype={"media_id": str, "id": str})
pilot["media_id"] = pilot["media_id"].str.strip()

def load_texts(pilot: pd.DataFrame) -> pd.Series:
    lookup, missing_files = {}, []
    for user in pilot["username"].unique():
        f = META_DIR / f"{user}-FullVideoInfo.csv"
        if not f.exists():
            missing_files.append(user); continue
        meta = pd.read_csv(f, dtype={"media_id": str})
        meta["media_id"] = meta["media_id"].str.strip()
        meta = meta.drop_duplicates("media_id")
        lookup.update(zip(meta["media_id"], meta["text"]))
    if missing_files:
        warnings.warn(f"找不到 metadata 檔：{missing_files}")
    return pilot["media_id"].map(lookup)

pilot["text_raw"] = load_texts(pilot)
n_missing = pilot["text_raw"].isna().sum()
print(f"共 {len(pilot)} 篇，找不到文字 {n_missing} 篇")
pilot["text_raw"] = pilot["text_raw"].fillna("").astype(str)


共 299 篇，找不到文字 0 篇


## 業配詞刪除 (待確認)

**程式邏輯**
1. **`normalize()`：先整理文字**
   - `NFKC` 正規化：全形轉半形（`＃業配` → `#業配`），不然後面抓不到
   - 刪掉網址：網址沒有語意，還會占用 token 名額
   - 把連續空白縮成一個
2. **`build_patterns()`：寫出四種「要抓的文字長相」**（正則表達式）
   - `#[^\s#]*(?:業配|工商|...)[^\s#]*`：任何**含有**業配詞的整個 hashtag，例如 `#XX品牌業配`
   - `#(?:ad|sponsored|...)\b`：英文 hashtag。`\b` 是「單字邊界」，所以 `#ad` 會被抓，`#adidas` 不會
   - `paid\s+partnership` 等英文片語（`\s+` 代表中間可以有一個或多個空白）
   - 最後，文中單獨出現的中文業配詞
3. **`mask_sponsor_terms()`**：依序套用這四種規則，`subn` 會把抓到的換成空白，並回傳「換了幾處」

**洩漏檢查怎麼看？**
輸出的表格裡：
- `mean`：該組有被刪到字的貼文比例
- `sum`：有被刪到字的篇數
- `count`：該組總篇數

如果業配組接近 100%、非業配組接近 0%，代表標籤幾乎就是看這些詞貼的，洩漏很嚴重。

> **小字典｜label 0 / 1**：0 = 非業配，1 = 業配。

In [ ]:
# Tier 1：明確揭露詞 → 遮
ZH_STRICT = ["業配", "工商", "廣告", "贊助", "置入", "付費合作", "品牌合作",
             "合作邀約", "邀約", "體驗邀請", "公關品", "公關邀約", "折扣碼", "優惠碼","乾爹", "廠商"]
EN_HASHTAG = ["ad", "ads", "advert", "advertisement", "sponsored", "sponsor",
              "sponsorship", "paidpartnership", "partner", "partnership",
              "collab", "gifted", "pr", "promo"]
EN_PHRASE  = [r"paid\s+partnership", r"sponsored\s+by", r"in\s+partnership\s+with",
              r"thanks\s+to\s+\S+\s+for\s+sponsoring"]
OTHER_LANG = ["협찬", "광고", "タイアップ"]          # 偶爾夾雜的韓、日文
# Tier 2：模糊詞 → 預設不遮，ablation 時可加入
ZH_OPTIONAL = ["合作", "感謝", "團購", "買"]

def build_patterns(zh_terms):
    zh = "|".join(map(re.escape, sorted(zh_terms, key=len, reverse=True)))
    return [
        re.compile(rf"#[^\s#]*(?:{zh})[^\s#]*"),                        # 含揭露詞的整個 hashtag
        re.compile(rf"#(?:{'|'.join(EN_HASHTAG)})\b", re.IGNORECASE),   # #ad 不會誤砍 #adidas
        re.compile(rf"\b(?:{'|'.join(EN_PHRASE)})\b", re.IGNORECASE),
        re.compile(rf"(?:{zh})"),                                       # 剩下的裸字
    ]

PATTERNS = build_patterns(ZH_STRICT + OTHER_LANG)

def normalize(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)        # 全形＃→#、全形英數→半形
    text = re.sub(r"https?://\S+", " ", text)         # 網址沒有語意，順便省 token
    return re.sub(r"\s+", " ", text).strip()

def mask_sponsor_terms(text: str):
    hits = 0
    for p in PATTERNS:
        text, n = p.subn(" ", text)
        hits += n
    return re.sub(r"\s+", " ", text).strip(), hits

pilot["text_norm"] = pilot["text_raw"].map(normalize)
masked = pilot["text_norm"].map(mask_sponsor_terms)
pilot["text_masked"] = masked.str[0]
pilot["mask_hits"]   = masked.str[1]

# 洩漏檢查：各 label 中被遮到字的貼文比例
print(pilot.assign(has_hit=pilot.mask_hits > 0)
           .groupby("label")["has_hit"].agg(["mean", "sum", "count"]))

# 隨機抽幾篇看前後差異
for _, r in pilot[pilot.mask_hits > 0].sample(min(5, (pilot.mask_hits > 0).sum()),
                                              random_state=0).iterrows():
    print("BEFORE:", r.text_norm[:150]); print("AFTER :", r.text_masked[:150]); print("-" * 60)

           mean  sum  count
label                      
0      0.006135    1    163
1      0.102941   14    136
BEFORE: 雜耍演員變世界第一健美先生💪|志祺七七 #shorts #promo 👊 medisana 56合1|智能八點傳感體脂計 體脂、BMI、心律、肌肉量、趨勢圖⋯ 等 56 項身體數據輕鬆掌握! ❗️連結點不到?從首頁點擊我們的 Portaly頁面也可以進入唷! ✔︎ 5.25吋彩色大螢幕|色彩鮮明清晰
AFTER : 雜耍演員變世界第一健美先生💪|志祺七七 #shorts 👊 medisana 56合1|智能八點傳感體脂計 體脂、BMI、心律、肌肉量、趨勢圖⋯ 等 56 項身體數據輕鬆掌握! ❗️連結點不到?從首頁點擊我們的 Portaly頁面也可以進入唷! ✔︎ 5.25吋彩色大螢幕|色彩鮮明清晰,數據一目瞭然
------------------------------------------------------------
BEFORE: 「今天我們要來開箱,泰國曼谷全新飯店!」 - 前兩年來曼谷玩的時候,一直就看到日本有名的西鐵集團要進駐啦!終於在2024年9月開幕,位於超級熱鬧的silom區,出門不用3分鐘就可以走到地鐵站,超級方便! - 說實話入住前超級緊張,上網刷到滿滿負評,有人說水壓不夠、有人說半夜很吵、有人說水管有臭味,但
AFTER : 「今天我們要來開箱,泰國曼谷全新飯店!」 - 前兩年來曼谷玩的時候,一直就看到日本有名的西鐵集團要進駐啦!終於在2024年9月開幕,位於超級熱鬧的silom區,出門不用3分鐘就可以走到地鐵站,超級方便! - 說實話入住前超級緊張,上網刷到滿滿負評,有人說水壓不夠、有人說半夜很吵、有人說水管有臭味,但
------------------------------------------------------------
BEFORE: 我送我老公的聖誕禮物...你們猜他最後有收嗎😳? 六萬塊的電腦捨得花~當然眼睛也要認真顧! 懷孕、備孕、全家大小都能吃的昕顏靈光凍再次分享給你們🧡 超多人反饋吃一週就有感~真的不是開玩笑😎 微米分子更好吸收,這就是為什麼吃了真的有效! 現在昕顏

## 載入模型

**※BERT 是什麼？**

**BERT**（Bidirectional Encoder Representations from Transformers）是 Google 在 2018 年發表的語言模型。它事先讀過大量文章，練習「克漏字」：把句子挖掉一個字，猜被挖掉的是什麼。練久了，它就學會了中文字和字之間的關係。

BERT 最重要的能力是**看上下文**。同樣是「蘋果」：
- 「蘋果**手機**」裡的蘋果 → 科技公司
- 「蘋果**汁**」裡的蘋果 → 水果

BERT 給這兩個「蘋果」的向量會不一樣，因為它會同時看前後文（所以叫 Bidirectional，雙向）。

---

**※步驟一：Tokenizer — 把文字切成小塊**

模型不能直接吃文字，要先切成一塊一塊的 **token**，再把每個 token 換成它在字典裡的編號（`input_ids`）。

`bert-base-chinese` 的切法：
- **中文：一個字一個 token**。「保健食品」→ `保` `健` `食` `品`
- **英文：切成常見的字根**。字典裡沒有的長單字會被切成幾塊，接在後面的那一塊開頭會加 `##`
- **字典裡完全沒有的符號**（例如很多 emoji）→ `[UNK]`（unknown，未知）

切完之後，tokenizer 還會自動在**最前面**加一個特殊 token：**`[CLS]`**（在最後面加 `[SEP]`，論文沒寫，但實際上有）：

```
原文：   這 款 面 膜 好 用
切完：   [CLS] 這 款 面 膜 好 用 [SEP]
編號：   101  6857 3621 7481 5606 1962 4500 102   ← input_ids（數字僅示意）
```

**※一批貼文長度不同怎麼辦？**
- **padding（補齊）**：短的貼文在後面補 `[PAD]`，補到和這一批最長的一樣長
- **truncation（截斷）**：超過 512 的切掉
- **attention_mask**：一串 0 和 1，告訴模型哪些是真的字（1）、哪些只是補位用的 `[PAD]`（0），補位的不要理

---

**※ 步驟二：[CLS] 是什麼？為什麼只拿它？**

`[CLS]` 是 classification（分類）的縮寫。它是一個**本身沒有意思的空位**，專門用來裝「整句話的摘要」。

**比喻：班長。** 全班（每個字）討論的時候，班長會聽每個人說話；討論結束後，老師只要問班長，就能知道全班的結論。`[CLS]` 就是這個班長。因為 BERT 裡每個 token 都會看所有其他 token，所以放在第 0 號位置的 `[CLS]` 最後會帶有整篇貼文的資訊。

論文寫「`[CLS]` 可以減輕位置偏差」的意思是：如果拿第一個**真正的字**當代表，它會偏向那個字本身的意思；`[CLS]` 自己沒有意思，比較能公平地代表整句話。

---

**※ 步驟三：Transformer 一層一層「討論」**

BERT-base 有 **12 層** Transformer。每一層裡，每個 token 都會：
1. 看看其他所有 token（這個機制叫 **self-attention，自注意力**）
2. 決定要多參考誰，例如「面膜」會多看「好用」
3. 更新自己的向量（768 個數字）

12 層就像開了 12 輪討論，每一輪大家都更懂這段話。

---

**※ 論文公式逐一拆解**

**① 貼文與文字**

$$p_n:\ \text{第 } n \text{ 篇貼文}, \qquad t_n:\ \text{這篇貼文的文字}$$

**② 原始的字詞序列（共 $j$ 個字）**

$$c = [c_1, c_2, \cdots, c_j]$$

**③ 切 token 並在最前面加上 [CLS]**

$$c' = [\,\text{[CLS]},\ c'_1,\ c'_2,\ \cdots,\ c'_{j'}\,]$$

- 加撇號（$'$）是因為**切完 token 之後的單位和原本的「字詞」不一樣了**，例如一個英文字可能被切成好幾塊，所以長度也從 $j$ 變成 $j'$

**④ $n$ 篇貼文一起送進模型**

$$\mathbb{Q}^{(j'+1)\times n}$$

- 意思是：$n$ 篇貼文，每篇有 $j'+1$ 個 token（$+1$ 就是多出來的 [CLS]）
- 對應程式裡 tokenizer 產生的 `input_ids`，形狀是 `(n, j'+1)`（行列順序和論文相反，但意思一樣）
- 這個符號是論文作者自己的寫法，不是通用的數學記號，看懂「$n$ 篇 × 每篇 $j'+1$ 個 token」就夠了

**⑤ 一層一層往上算**

$$\mathbf{F}_T\!\left(\mathbf{F}_T^{\,o-1}\right) = D^{o} = \left[\, d^{o}_{\text{[CLS]}},\ d^{o}_{1},\ d^{o}_{2},\ \cdots,\ d^{o}_{j'} \,\right]$$

| 符號 | 白話 |
|---|---|
| $\mathbf{F}_T$ | 整個 Transformer（下標 $T$ = Text） |
| $o$ | 總層數；BERT-base 的 $o = 12$ |
| $\mathbf{F}_T^{\,o-1}$ | 第 $o-1$ 層（第 11 層）的輸出 |
| $\mathbf{F}_T(\mathbf{F}_T^{\,o-1})$ | 把第 11 層的結果再送進最後一層 |
| $D^{o}$ | 最後一層（第 12 層）的輸出：**每個 token 各一個向量** |
| $d^{o}_{\text{[CLS]}}$ | 最後一層裡，[CLS] 那個位置的向量 |
| $d^{o}_{1}, \cdots, d^{o}_{j'}$ | 最後一層裡，每個字各自的向量 |

換成比較好懂的「遞迴」寫法，第 $l$ 層的輸出就是把前一層的輸出再加工一次：

$$D^{0} = \text{Embedding}(c'), \qquad D^{l} = \text{Layer}_l\!\left(D^{l-1}\right),\quad l = 1, 2, \ldots, 12$$

**⑥ 只拿 [CLS] 當整篇貼文的代表**

$$X^{T} = d^{o}_{\text{[CLS]}} \in \mathbb{R}^{768}$$

- $\mathbb{R}^{768}$：由 768 個實數組成的向量
- 上標 $T$ = Text，之後還會有影片的 $X^{V}$

**※公式 ↔ 程式對照**

| 論文 | 程式 |
|---|---|
| $c'$（加了 [CLS] 的 token 序列） | `tokenizer(...)["input_ids"]` |
| $\mathbf{F}_T$ | `model`（BertModel） |
| $D^{o}$ | `model(**batch).last_hidden_state`，形狀 `(B, L, 768)` |
| $d^{o}_{\text{[CLS]}}$ | `last_hidden_state[:, 0]`（第 0 號位置） |
| $X^{T}$ | `X_T`，形狀 `(299, 768)` |

In [21]:
if MODEL_NAME.startswith("ckiplab/"):
    tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")  # CKIP 官方指定
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

texts = (pilot["text_masked"] if APPLY_MASK else pilot["text_norm"]).tolist()

# 看有多少貼文會被截斷
lens = [len(tokenizer(t, add_special_tokens=True)["input_ids"]) for t in texts]
print(f"token 長度 中位數={np.median(lens):.0f}, P95={np.percentile(lens,95):.0f}, "
      f"超過 {MAX_LEN} 的比例={np.mean(np.array(lens) > MAX_LEN):.1%}")
print("[UNK] 比例：",
      np.mean([tokenizer(t)["input_ids"].count(tokenizer.unk_token_id) / max(len(tokenizer(t)["input_ids"]), 1)
               for t in texts[:200]]).round(4))


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ckiplab/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] Token indices seque

token 長度 中位數=52, P95=433, 超過 512 的比例=2.7%
[UNK] 比例： 0.0508


## 抽 CLS embedding

```python
@torch.no_grad()                      # ① 不記錄梯度
def encode_cls(texts, batch_size):
    for i in range(0, len(texts), batch_size):          # ② 每 32 篇一批
        batch = tokenizer(..., padding=True,             # ③ 切字、補齊、截斷
                          truncation=True, max_length=512,
                          return_tensors="pt").to(device)
        h = model(**batch).last_hidden_state             # ④ 跑 12 層 → D^o
        out.append(h[:, 0])                              # ⑤ 只拿 [CLS]
    return torch.cat(out)                                # ⑥ 把每一批接起來
```

1. **`@torch.no_grad()`**：訓練模型時，PyTorch 會記錄每一步計算，以便之後回頭修正（這叫「梯度」）。我們現在只是**用**模型、不是**訓練**它，關掉記錄可以省很多記憶體、跑得更快
2. **分批**：299 篇 ÷ 32 = 10 批，就是進度條上的 `0/10`
3. **`return_tensors="pt"`**：輸出 PyTorch 的 tensor 格式
4. **`model(**batch)`**：`**` 是把字典拆開傳進去，等於 `model(input_ids=..., attention_mask=...)`
   - `last_hidden_state` 的形狀是 **`(B, L, 768)`**：B 篇貼文 × 每篇 L 個 token × 每個 token 768 個數字
5. **`h[:, 0]`**：`:` 代表「每一篇都要」，`0` 代表「只要第 0 號位置」，也就是 [CLS] → 形狀變成 `(B, 768)`
6. **`torch.cat`**：把 10 批接成一個大表格 → **`(299, 768)`**

`torch.autocast`：在 GPU 上自動改用半精度（16 位元）計算，速度更快；CPU 上不啟用。

In [22]:
@torch.no_grad()
def encode_cls(texts, batch_size=BATCH_SIZE):
    out = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                          max_length=MAX_LEN, return_tensors="pt").to(device)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            h = model(**batch).last_hidden_state          # (B, L, 768)
        out.append(h[:, 0].float().cpu())                 # 取位置 0 = [CLS]，不用 pooler_output
    return torch.cat(out)

X_T = encode_cls(texts)
print(X_T.shape)   # (N, 768)

  0%|          | 0/10 [00:00<?, ?it/s]

torch.Size([299, 768])


## 存檔（以 media_id 對齊之後的 video embedding）

用 `torch.save` 把向量和 `media_id`、標籤一起存成 `.pt` 檔。之後影片的向量也會用 `media_id` 對齊，才能把同一篇貼文的文字和影片配在一起。

**※ Linear Probe：這些向量有用嗎？**
在向量上面接一個**最簡單的分類器**（邏輯迴歸，Logistic Regression），看它能不能分辨業配和非業配。如果連簡單的分類器都分得出來，代表向量裡確實裝了有用的資訊。

**`StratifiedGroupKFold`（分組交叉驗證）**
- 把資料分成 5 份，輪流用其中 4 份訓練、剩下 1 份測試，總共考 5 次
- **Group**：同一位網紅的貼文只會出現在同一份裡。不然模型可能只是記住「這位網紅很常接業配」，而不是學會看內容
- **Stratified**：每一份裡業配和非業配的比例盡量一致

**ROC-AUC 怎麼讀？**
- 隨機抽一篇業配和一篇非業配，模型給業配那篇比較高分的機率
- 0.5 = 跟丟銅板一樣；1.0 = 完美
- 目前結果 **0.787 ± 0.104**：只靠文字、而且業配詞已經刪掉，仍然有相當的辨識力。不過 ±0.104 的波動不小，因為 pilot 只有 299 篇，再分給 5 個 fold，每一份的網紅組成差很多

In [23]:
tag = f"{MODEL_NAME.split('/')[-1]}_{'masked' if APPLY_MASK else 'raw'}"
torch.save({"media_id": pilot["media_id"].tolist(),
            "label": pilot["label"].tolist(),
            "X_T": X_T,
            "model": MODEL_NAME, "masked": APPLY_MASK, "max_len": MAX_LEN},
           OUT_DIR / f"pilot_text_{tag}.pt")
pilot[["media_id", "username", "label", "mask_hits", "text_masked"]].to_csv(
    OUT_DIR / f"pilot_text_{tag}.csv", index=False)
print("saved:", OUT_DIR / f"pilot_text_{tag}.pt")

# %% Cell 7 ── Sanity check：linear probe（依網紅分組做 CV，避免同一網紅跨 fold）
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

y, groups = pilot["label"].values, pilot["username"].values
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
auc = cross_val_score(LogisticRegression(max_iter=2000), X_T.numpy(), y,
                      groups=groups, cv=cv, scoring="roc_auc")
print(f"[{tag}] linear probe ROC-AUC = {auc.mean():.3f} ± {auc.std():.3f}")
# 解讀：masked 和 raw 各跑一次。raw 的 AUC 如果明顯高出很多（例如接近 1.0），
# 代表揭露詞確實在洩漏標籤，論文裡也可以拿這組對照來說明遮蔽的必要性。

saved: /home/yingtzudai/Multimodal_SPoD_Video/features/text/pilot_text_bert-base-chinese_masked.pt
[bert-base-chinese_masked] linear probe ROC-AUC = 0.787 ± 0.104


## (之後端到端訓練用）可 fine-tune 的 Text Encoder module

前面的做法是 **freeze（凍結）**：BERT 完全不動，只拿它算出來的向量。之後正式訓練 SPoD 時，可以選擇讓 BERT 跟著一起微調（**fine-tune**），讓它更懂「業配語氣」。

- `nn.Module`：PyTorch 裡所有模型零件的基本模板。只要定義兩件事：
  - `__init__`：這個零件裡有什麼（這裡是一個 BERT）
  - `forward`：資料進來後怎麼算（跑 BERT → 取 [CLS]）
- `requires_grad = False`：把參數鎖住，訓練時不會被改動，也就是凍結
- `out_dim`：輸出的維度（768），之後接其他模組時會用到

| | Freeze（凍結） | Fine-tune（微調） |
|---|---|---|
| BERT 參數 | 不動 | 跟著任務一起調整 |
| 速度、記憶體 | 快、省 | 慢、需要 GPU |
| 資料量需求 | 少量就行 | 需要較多資料，否則容易過度擬合 |
| 適合 | pilot、快速實驗 | 正式模型 |

In [24]:
import torch.nn as nn

class TextEncoder(nn.Module):
    """F_T：輸出 [CLS] 表示 X^T。freeze=True 對應 pilot 的特徵抽取設定。"""
    def __init__(self, name=MODEL_NAME, freeze=False):
        super().__init__()
        self.bert = AutoModel.from_pretrained(name)
        if freeze:
            for p in self.bert.parameters():
                p.requires_grad = False
        self.out_dim = self.bert.config.hidden_size

    def forward(self, input_ids, attention_mask):
        return self.bert(input_ids=input_ids,
                         attention_mask=attention_mask).last_hidden_state[:, 0]